# 1B Stream Responses and Measure Performance

**Purpose:**
See how fast your endpoint answers, from the user's point of view and from the GPU's.

**What we're doing**

We stream a reply token by token, time how long the first word takes, and then send several requests at once to see how vLLM shares one GPU between them.

**Why we're doing it**

* Streaming is how chat applications feel fast, even when the full answer takes seconds
* Latency and throughput are different numbers, and you need both to size a deployment
* vLLM batches concurrent requests on the GPU. Seeing that happen explains why one small instance can serve several users

**What to look for**

* The gap between the time to the first token and the total time
* Total tokens per second rising as you add concurrent requests, while each request slows down far less than you might expect
* The same traffic on the instance dashboard's **Tokens per second** and **Latency** panels

**Output**

Time to first token, generation speed, and a small concurrency table for your endpoint.

> **Prerequisite:** Section 1 is complete: the `qwen-lab` instance is **Active** and `.env` exists at the project root.


## 1B.1 Load Values from `.env`

Run the cell below. It reads the `.env` file you created in Section 1.6, so the endpoint, model name, and token are the same ones you used there.

In [ ]:
import sys
sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as base_url, KLIQUE_MODEL as model, KLIQUE_GATEWAY_TOKEN as token
import requests

url_models = f"{base_url}/models"
url_chat = f"{base_url}/chat/completions"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

print(f"Endpoint: {base_url}")
print(f"Model:    {model}")
print(f"Token:    {token[:8]}...")

## 1B.2 Stream a Reply

In Section 1.8.2 the cell waited for the whole answer, then printed it. With `"stream": True`, vLLM sends each piece of the answer as soon as it is generated, using **server-sent events**: a stream of lines that start with `data: `, each holding one small JSON chunk, and a final `data: [DONE]`.

Each chunk carries the new text in `choices[0].delta.content`. Setting `stream_options.include_usage` adds one last chunk with the token counts, which a stream does not otherwise report.

The function below sends one streamed request, prints the text as it arrives, and records three times: when the request was sent, when the first text arrived, and when the stream ended. `requests` reads the stream because the call passes `stream=True`.

In [ ]:
import json
import time


def stream_chat(prompt, max_tokens=256, show=True):
    """Stream one chat completion. Returns its timings and token counts, or None on an error."""
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "stream_options": {"include_usage": True},
        "chat_template_kwargs": {"enable_thinking": False},
    }
    start = time.perf_counter()
    first = None
    usage = None
    finish_reason = None
    with requests.post(url_chat, headers=headers, json=data, stream=True, timeout=120) as response:
        if not response.ok:
            print(f"❌ HTTP {response.status_code}: {response.text[:300]}")
            return None
        for raw in response.iter_lines():
            line = raw.decode("utf-8")
            if not line.startswith("data: "):
                continue
            payload = line[len("data: "):]
            if payload == "[DONE]":
                break
            chunk = json.loads(payload)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for choice in chunk.get("choices", []):
                text = choice.get("delta", {}).get("content") or ""
                if text and first is None:
                    first = time.perf_counter()
                if text and show:
                    print(text, end="", flush=True)
                finish_reason = choice.get("finish_reason") or finish_reason
    end = time.perf_counter()
    if show:
        print()
    return {
        "ttft": (first if first is not None else end) - start,
        "total": end - start,
        "completion_tokens": usage["completion_tokens"] if usage else None,
        "first": first,
        "end": end,
        "finish_reason": finish_reason,
    }


result = stream_chat("Explain in one paragraph why a GPU is good at serving language models.")

You should see the paragraph appear a few words at a time instead of all at once.

> **Checkpoint:** text prints progressively and no `❌` line appears. If the whole answer appears at once after a pause, see the Troubleshooting table at the end.

## 1B.3 Time to First Token and Generation Speed

Two numbers describe how a single reply feels:

| Measure | What it means | What drives it |
|---|---|---|
| **Time to first token (TTFT)** | How long the user waits before anything appears | Network, the gateway, queueing, and reading the prompt (*prefill*) |
| **Generation speed** | Tokens per second once text is flowing (*decode*) | The GPU, the model size, and how many other requests share the GPU |

Generation speed below counts the tokens after the first one, over the time after the first one arrived.

In [ ]:
if result:
    print(f"Time to first token: {result['ttft']:.2f} s")
    print(f"Total time:          {result['total']:.2f} s")
    tokens = result["completion_tokens"]
    if tokens and tokens > 1 and result["first"] is not None:
        print(f"Completion tokens:   {tokens}")
        print(f"Generation speed:    {(tokens - 1) / (result['end'] - result['first']):.1f} tokens/s")
    else:
        print("The stream did not report usage, so generation speed is unavailable.")
    print(f"Finish reason:       {result['finish_reason']}")

Run the cell pair a few times. The first request after a quiet period can be slower; later ones settle.

Now compare a **non-streamed** request for the same prompt. The model does the same work, so the total time is similar. The difference is that the user sees nothing until it is all done: for a non-streamed request, the time to first token *is* the total time.

In [ ]:
data = {
    "model": model,
    "messages": [{"role": "user", "content": "Explain in one paragraph why a GPU is good at serving language models."}],
    "max_tokens": 256,
    "chat_template_kwargs": {"enable_thinking": False},
}
start = time.perf_counter()
response = requests.post(url_chat, headers=headers, json=data, timeout=120)
elapsed = time.perf_counter() - start
if response.ok:
    print(f"Non-streamed total time: {elapsed:.2f} s for {response.json()['usage']['completion_tokens']} tokens")
    if result:
        print(f"Streamed first token:    {result['ttft']:.2f} s")
else:
    print(f"❌ HTTP {response.status_code}: {response.text[:300]}")

## 1B.4 Send Requests Concurrently

A GPU generating for one request is mostly waiting on memory, not doing arithmetic. vLLM uses that headroom with **continuous batching**: it adds new requests to the batch on every step instead of finishing one request before starting the next. Several requests at once therefore cost far less than several requests one after another.

The cell below sends a small, fixed number of identical requests at three concurrency levels: 1, then 4, then 8 at the same time, 13 requests in total. Each asks for exactly `MAX_TOKENS` tokens. `ignore_eos` is a vLLM-specific option that keeps generating past the model's natural end, so every request does the same amount of work and the numbers are comparable.

> **Note:** If others share this instance or its GPU, their traffic affects your numbers. Keep the levels small; this is a demonstration, not a load test.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

LEVELS = [1, 4, 8]
MAX_TOKENS = 128
PROMPT = "Write a short paragraph about how GPUs speed up model serving."


def timed_request(_):
    """Send one non-streamed request. Returns (seconds, completion tokens, HTTP status or error name)."""
    data = {
        "model": model,
        "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": MAX_TOKENS,
        "ignore_eos": True,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    start = time.perf_counter()
    try:
        response = requests.post(url_chat, headers=headers, json=data, timeout=180)
    except requests.RequestException as error:
        return time.perf_counter() - start, 0, type(error).__name__
    elapsed = time.perf_counter() - start
    tokens = response.json()["usage"]["completion_tokens"] if response.ok else 0
    return elapsed, tokens, response.status_code


rows = []
for n in LEVELS:
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n) as pool:
        results = list(pool.map(timed_request, range(n)))
    wall = time.perf_counter() - start
    errors = [status for _, _, status in results if status != 200]
    tokens = sum(t for _, t, _ in results)
    avg_latency = sum(e for e, _, _ in results) / n
    rows.append((n, wall, avg_latency, tokens / wall, errors))

print(f"{'Concurrent':>10} | {'Wall time':>9} | {'Avg latency':>11} | {'Total tokens/s':>14}")
for n, wall, avg_latency, throughput, errors in rows:
    note = f"  ❌ {sorted(set(map(str, errors)))}" if errors else ""
    print(f"{n:>10} | {wall:>8.1f}s | {avg_latency:>10.1f}s | {throughput:>14.1f}{note}")

Read the table row by row:

* **Avg latency** is how long each request took. It rises as you add requests, because they share the GPU.
* **Total tokens/s** is all the tokens generated, divided by the wall time. On most GPUs it rises much faster than latency does: eight requests at once finish in far less than eight times the time of one.

That gap is continuous batching at work. It is also the trade-off you tune when sizing a deployment: more concurrent requests mean more total throughput but a slower experience for each user. Section 1D shows the vLLM arguments that cap it.

> **Checkpoint:** three rows with no `❌`, and total tokens/s at 8 concurrent is clearly higher than at 1.

## 1B.5 Compare with the Dashboard

Open the `qwen-lab` instance dashboard from Section 1.4. The **Total number of requests** panel now includes these calls, and the **Tokens per second** and **Latency** panels show the burst you just sent.

<!-- screenshot: images/dashboard_concurrency.png (Tokens per second and Latency panels after the 1B.4 burst) -->

The dashboard measures at the server, so its latency does not include the network between this notebook and the endpoint. Your notebook numbers are what a client actually experiences. Both are useful: if the dashboard is fast and your client is slow, look at the network, not the GPU.

**If you are continuing to Section 1C or Section 2, leave the instance running.** Otherwise, stop it as described in Section 1.9.


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| The whole answer appears at once after a pause | Something between you and the endpoint, such as a proxy, buffers the stream. The request still works; TTFT then equals the total time. |
| `KeyError` or `None` for completion tokens | The stream did not include a usage chunk. Check that the request sets `stream_options.include_usage`. |
| `400` mentioning `ignore_eos` | The server rejected a vLLM-specific option. Remove `"ignore_eos": True`; the requests then end at different lengths, so compare tokens/s rather than latency. |
| `429` or `503` at higher concurrency | A rate limit or a busy server. Lower `LEVELS`, wait, and run the cell again. |
| Throughput barely rises with concurrency | The GPU is already busy with other traffic, or the instance is limited to very few concurrent sequences (see Section 1D). |
| Connection error or timeout | The instance stopped, often from the idle time limit set in 1.3. Check its status and relaunch it. |

## Next: Constrain What the Model Returns

You now know how fast the endpoint is. Next, make its output reliable enough for a program to use.

**Next:** In `01c_Structured_Output_and_Tool_Calling.ipynb` we make the model return JSON that matches a schema, and let it call a Python function.